# Deep Learning Appraoch to Out of Field Artifact Removal in 3rd Generation CT-Scans

## General Functions

In [1]:
import stackview as sv
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import os

In [2]:
def read_raw_image(filepath, dtype, shape = (512, 512)):
    data = np.fromfile(filepath, dtype=dtype)
    return data.reshape(shape)

In [3]:
def normalize_img(img):
    img = img.astype(np.float32)  # ensure float
    img_norm = (img - img.min()) / (img.max() - img.min())
    return img_norm

In [4]:
def read_raw_scan(scan, height = 512, width = 512):
    height = height
    width = width
    dtype = np.uint16
    folder = "../../raw/" + scan
    files = os.listdir('../../raw/' + scan)
    files = sorted(files, key=lambda x: int(x.split('_')[1].split('.')[0]))
    slices = []

    for f in files:
        if f.endswith(".raw"):
            img = read_raw_image(
                os.path.join(folder, f),
                shape=(height, width),
                dtype=dtype
            )
            img = normalize_img(img)
            slices.append(img)

    volume = np.stack(slices, axis=0)
    return volume

def read_alt_scan(scan, type = 'altered', height = 512, width = 512):
    height = height
    width = width
    dtype = np.uint16
    folder = f"../../{type}/" + scan
    files = os.listdir(f'../../{type}/' + scan)
    files = sorted(files, key=lambda x: int(x.split('_')[1].split('.')[0]))
    slices = []

    for f in files:
        if f.endswith(".raw"):
            img = read_raw_image(
                os.path.join(folder, f),
                shape=(height, width),
                dtype=np.float32
            )
            img = normalize_img(img)
            slices.append(img)

    volume = np.stack(slices, axis=0)
    return volume

In [5]:
def create_side_by_side(vol_1, vol_2):
    side_by_side = []

    for orig, den in zip(vol_1, vol_2):
        combined = np.concatenate([orig, den], axis=1)  # horizontal stack
        side_by_side.append(combined)

    side_by_side = np.stack(side_by_side, axis=0)
    return side_by_side

In [6]:
base_raw_path = '../../raw/'
base_alt_path = '../../altered/'
base_res_path = '../../resnet/'
base_unt_path = '../../unet/'
raw_scan_dirs = sorted(os.listdir(base_raw_path))
raw_scan_dirs.pop()
alt_scan_dirs = sorted(os.listdir(base_alt_path))
res_scan_dirs = sorted(os.listdir(base_res_path))
unt_scan_dirs = sorted(os.listdir(base_unt_path))

## Problem Description

* CT-Scans are very useful for diagnostic purposes
* Modern CT scan use an X-ray emitter and an X-ray detector that rotate around the patient
* As the x-ray emitter rotates around the patient, the patient is pushed through the field of view
* We want to create as clear of a picture as possible with the smallest x-ray dose possible

In [7]:
# Transverse and Saggital Views of a CT-Scan
from IPython.display import display, HTML

display(HTML("""
<div style="
    display: grid;
    grid-template-columns: max-content max-content;
    gap: 0px;
    align-items: start;
    justify-content: start;
">

  <!-- Tall image (left) -->
  <div>
    <img src="imgs/3rd_gen_img.jpg" style="display: block; height: 500px; width: auto;">
  </div>

  <!-- Wide image (right, snug) -->
  <div>
    <img src="imgs/saggital_plane_of_ct_scan.jpg" style="display: block; height: 400px; width: auto;">
  </div>

</div>
"""))

* This is an abdominal scan of a patient viewed through the transverse plane
* Created using 3rd Gen CT-scan and reconstructed through Filtered Back Projection

In [8]:
# Good Scan vs Bad Scan
good_scan = 'L008'
good_raw_scan = read_raw_scan(good_scan)
bad_scan = 'L123'
bad_raw_scan = read_raw_scan(bad_scan)
combo = create_side_by_side(good_raw_scan, bad_raw_scan)
print('Scan L008 Without OOF Artifact \t\t\t\t\t Scan L123 With 00F Artifact')
sv.slice(combo, slice_number=0)

Scan L008 Without OOF Artifact 					 Scan L123 With 00F Artifact


* We want to reduce these out of field artifacts without producing hallucinations

## Generating OOF Artifacts

* Take original scans and add streaks similar to the ones found in scan L123
* Orignal scans are the desired output
* Corrupted scans are the input

In [9]:
# Original Scan vs Output Scan
scan = 'L008'
raw_scan = read_raw_scan(scan)
alt_scan = read_alt_scan(scan)
combo = create_side_by_side(raw_scan, alt_scan)
print('\t Scan L008 Without Simulated OOF Artifact \t\t\t Scan L008 With Simulated 00F Artifact')
sv.slice(combo, slice_number=0)

	 Scan L008 Without Simulated OOF Artifact 			 Scan L008 With Simulated 00F Artifact


## Model Training

* 2 models were tested
    * Standard U-Net architecture (10 epochs)
        * Should directly predict the image
    * Residual U-net architecture (20 epochs)
        * Predicts only where the streaks are
        * Remove those streaks from the original image later
* Training Input
    * An individual slice/image from the ct-scan

## Model Output

In [10]:
# ResNet Comparisons
scan = 'L061'
raw_scan = read_raw_scan(scan)
alt_scan = read_alt_scan(scan)
new_scan = read_alt_scan(scan, type = 'resnet')
combo = create_side_by_side(alt_scan, new_scan)
combo = create_side_by_side(combo, raw_scan)
print(f'\t Scan {scan} With Simulated OOF Artifact \t\t\t\t Reconstructed Scan {scan} With Res-Net \t\t\t\t Original Scan {scan} ')
sv.slice(combo, slice_number=0)

	 Scan L061 With Simulated OOF Artifact 				 Reconstructed Scan L061 With Res-Net 				 Original Scan L061 


In [11]:
# ResNet Differences
alt_new_diff = new_scan - alt_scan
raw_new_diff = raw_scan - new_scan
sim_new_diff = raw_scan - alt_scan
diff_combo = create_side_by_side(sim_new_diff, alt_new_diff)
diff_combo = create_side_by_side(diff_combo, raw_new_diff)
print(f'\t Original {scan} - Simulated {scan} (Desired Difference) \t ResNet {scan} - Simulated {scan} (Actual difference) \t\t ResNet {scan} - Original {scan}')
sv.slice(diff_combo, slice_number=0)

	 Original L061 - Simulated L061 (Desired Difference) 	 ResNet L061 - Simulated L061 (Actual difference) 		 ResNet L061 - Original L061


In [12]:
# Unet Comparisons
raw_scan = read_raw_scan(scan)
alt_scan = read_alt_scan(scan)
new_scan = read_alt_scan(scan, type = 'unet')
combo = create_side_by_side(alt_scan, new_scan)
combo = create_side_by_side(combo, raw_scan)
print(f'\t Scan {scan} With Simulated OOF Artifact \t\t\t\t Reconstructed Scan {scan} With U-Net \t\t\t\t Original Scan {scan} ')
sv.slice(combo, slice_number=0)

	 Scan L061 With Simulated OOF Artifact 				 Reconstructed Scan L061 With U-Net 				 Original Scan L061 


In [13]:
# U-Net Differences
alt_new_diff = new_scan - alt_scan
raw_new_diff = raw_scan - new_scan
sim_new_diff = raw_scan - alt_scan
diff_combo = create_side_by_side(sim_new_diff, alt_new_diff)
diff_combo = create_side_by_side(diff_combo, raw_new_diff)
print(f'\t Original {scan} - Simulated {scan} (Desired Difference) \t U-Net {scan} - Simulated {scan} (Actual difference) \t\t U-Net {scan} - Original {scan}')
sv.slice(diff_combo, slice_number=0)

	 Original L061 - Simulated L061 (Desired Difference) 	 U-Net L061 - Simulated L061 (Actual difference) 		 U-Net L061 - Original L061


## Results

* U-Net
    * Marginally effective at reducing OOF artifacts
    * Doesn't hallucinate as bad as other models
* Res-Net
    * Very good at finding the OOF artifacts and streaks
    * Can hallucinate some parts of the scan

## Future Directions

* More realistic OOF artifact simulation
* Find or develop metric to examine levels of hallucinations
* Models that don't look at just one slice of the scan but the entire 3d object
* Adjusting Res-Net to be more accurate at finding streaks